In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col

LANDING_PATH = "/Volumes/dev/cdc/cdc_landing/"

@dp.table(name="bronze_orders_cdc", comment="Raw CDC changelog events")
def bronze_orders_cdc():
    return (spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "json")
            .load(LANDING_PATH))

dp.create_streaming_table("orders_current")

dp.create_auto_cdc_flow(
    target="orders_current",
    source="bronze_orders_cdc",
    keys=["order_id"],
    sequence_by=col("seq"),
    apply_as_deletes=col("op") == "DELETE",
    except_column_list=["op", "seq"],
    stored_as_scd_type=1,
)